[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/21_gradient_clipping_solution.ipynb)

# 🟢 Solution: Gradient Norm Clipping（参考版）

## 📋 题目描述

**难度：** Easy

实现梯度范数裁剪。

梯度裁剪在所有参数梯度的 L2 范数超过阈值时进行缩放，防止梯度爆炸。

**签名:** `clip_grad_norm(parameters, max_norm) -> float`

**参数:**
- `parameters` — 带 `.grad` 属性的张量列表
- `max_norm` — 允许的最大梯度范数

**返回:** 原始总梯度范数（浮点数）

**约束:**
- 总范数 = `sqrt(sum(p.grad.norm()^2))`
- 仅在总范数 > max_norm 时裁剪
- 保持梯度方向不变

**提示：** 1. total_norm = sqrt(Σ p.grad.norm()²)
2. clip_coef = max_norm / total_norm
3. 若 total_norm > max_norm：所有梯度乘以 clip_coef
4. 返回原始 total_norm（float）


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ SOLUTION

def clip_grad_norm(parameters, max_norm):
    # ---- Step 1: 过滤无梯度的参数 ----
    # 有些参数可能没有梯度（比如被冻结的参数、未使用的参数）
    # p.grad is None 检查是否有梯度
    parameters = [p for p in parameters if p.grad is not None]

    # ---- Step 2: 计算总梯度范数 ----
    # 所有参数的梯度拼在一起的 L2 范数
    # 数学公式：total_norm = sqrt(Σ ||p.grad||²)
    # ||p.grad|| = torch.norm(p.grad) 是该参数梯度的 L2 范数
    # 所有参数梯度的范数平方求和再开方 = 全局 L2 范数
    total_norm = torch.sqrt(sum(p.grad.norm() ** 2 for p in parameters))

    # ---- Step 3: 计算裁剪系数 ----
    # clip_coef = max_norm / total_norm
    # 如果 total_norm <= max_norm，clip_coef >= 1，不裁剪
    # 如果 total_norm > max_norm，clip_coef < 1，需要裁剪
    # 加 1e-6 防止除以零（total_norm 恰好为 0 的情况）
    clip_coef = max_norm / (total_norm + 1e-6)

    # ---- Step 4: 条件裁剪 ----
    # 只在 clip_coef < 1（即梯度范数超过阈值）时才裁剪
    # p.grad.mul_(clip_coef) 原地缩放所有梯度
    # 关键：所有梯度乘以同一个系数，保持梯度方向不变
    # 这样梯度下降的方向不变，只是步长变小了
    if clip_coef < 1:
        for p in parameters:
            p.grad.mul_(clip_coef)

    # ---- Step 5: 返回原始范数 ----
    # .item() 将标量张量转为 Python float
    return total_norm.item()


In [ ]:
p1 = torch.randn(3, requires_grad=True)
p2 = torch.randn(3, requires_grad=True)
(p1.sum() + p2.sum()).backward()
print('Before:', clip_grad_norm([p1, p2], max_norm=1.0))
print('p1 grad:', p1.grad)


In [ ]:
from torch_judge import check
check('gradient_clipping')


## 📝 核心思路总结

1. **全局范数**：所有参数梯度视为一个向量，计算其 L2 范数
2. **条件裁剪**：只在范数超过 max_norm 时缩放，否则不动
3. **方向不变**：所有梯度等比例缩放，保持下降方向
4. **数值安全**：加 epsilon 防止除以零
